# Competitive Programming Tag Predictor -- Qwen3-4B-Instruct-2507 + QLoRA
**Kaggle - GPU P100 ou T4 x2**

| | |
|---|---|
| Modelo base | `unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit` (4B params) |
| Tecnica | QLoRA -- NF4 4-bit + LoRA via Unsloth |
| Tarefa | Predicao multilabel de tags algoritmicas |
| Dataset | [Codeforces Competitive Programming Dataset](https://www.kaggle.com/datasets/dinuiongeorge/codeforces-competitive-programming-dataset) |
| Label masking | `train_on_responses_only` (Unsloth nativo -- sem bug de SFTTrainer) |

> Selecione **Settings -> Accelerator -> GPU P100 ou T4 x2** antes de rodar.

## Celula 1 -- Instalacao

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.55.2
!pip install --no-deps trl==0.22.2
!pip install scikit-learn

## Celula 2 -- Carregar modelo e QLoRA

**Qwen3-4B-Instruct-2507** carregado em NF4 4-bit via Unsloth.

`MAX_SEQ_LENGTH = 512`: cobre a grande maioria dos enunciados do Codeforces apos limpeza de LaTeX.

| Parametro LoRA | Valor | Obs |
|---|---|---|
| rank `r` | 32 | igual ao notebook base do Unsloth |
| `lora_alpha` | 32 | escala = 1.0 |
| `lora_dropout` | 0 | otimizado pelo Unsloth |
| `target_modules` | q/k/v/o + gate/up/down | atencao + MLP |
| `gradient_checkpointing` | `"unsloth"` | ~30% menos VRAM |

In [ ]:
SEED = 42

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel
import torch

MAX_SEQ_LENGTH = 512
base_model     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
adapter_path   = "/kaggle/input/datasets/sebascitta/qwen3-4b-instructor2507-top5trainingdataset/lora_adapter"

# 1. Carregar modelo base + tokenizer via Unsloth (única API correta para QLoRA)
#    AutoModelForCausalLM não é compatível com FastLanguageModel.for_training()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = base_model,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = True,
    dtype           = None,          # None → detecção automática (bf16/fp16)
)

# 2. Aplicar LoRA ao modelo base antes de carregar o adaptador anterior
model = FastLanguageModel.get_peft_model(
    model,
    r                 = 32,
    lora_alpha        = 32,
    lora_dropout      = 0,
    target_modules    = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state      = SEED,
    use_rslora        = False,
    loftq_config      = None,
)

# 3. Carregar pesos do adaptador LoRA do treino anterior
model = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)

# 4. Configurar template de chat do Qwen3 no tokenizer
#    Não passar chat_template='' aqui — get_chat_template cuida disso
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# 5. Ajustes finais
tokenizer.model_max_length = MAX_SEQ_LENGTH
model.config.use_cache     = False

print("✅ Modelo e adaptador prontos para continuar o treino.")
model.print_trainable_parameters()


## Celula 3 -- Chat template

O Qwen3 usa o template **ChatML** (`<|im_start|>` / `<|im_end|>`).
`get_chat_template` do Unsloth configura o tokenizer com o template correto,
incluindo os tokens especiais necessarios para `train_on_responses_only` funcionar.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)
print("JobDone")

## Celula 4 -- Pre-processamento do dataset

Dataset: `dinuiongeorge/codeforces-competitive-programming-dataset`

Colunas usadas: `problem_statement` (enunciado) e `problem_tags` (tags alvo).

Limpeza aplicada:
- Remove expressoes LaTeX (`$$$...$$$`, `$$...$$`, `\cmd{...}`) que poluem o vocabulario
- Remove tags de dificuldade (`*1800`, `*2200`, ...)
- Normaliza espacos e capitaliza em lower-case

O dataset e montado no formato `conversations` que o `apply_chat_template` do Unsloth espera:
```python
[{'role': 'user', 'content': enunciado},
 {'role': 'assistant', 'content': tags}]
```

In [ ]:
import pandas as pd
import re
from datasets import Dataset
from collections import Counter
import ast

# Caminhos dos arquivos (substitua pelo caminho real no Kaggle)
TRAIN_PATH = "/kaggle/input/datasets/dinuiongeorge/codeforces-competitive-programming-dataset/01_TASK_DATASETS/03_Task_Datasets/02_DATASETS_WO_TAG_ENCODING/OUR_DATASET/top_5_training_dataset.csv"
VAL_PATH   = "/kaggle/input/datasets/dinuiongeorge/codeforces-competitive-programming-dataset/01_TASK_DATASETS/03_Task_Datasets/02_DATASETS_WO_TAG_ENCODING/OUR_DATASET/top_5_validation_dataset.csv"
TEST_PATH  = "/kaggle/input/datasets/dinuiongeorge/codeforces-competitive-programming-dataset/01_TASK_DATASETS/03_Task_Datasets/02_DATASETS_WO_TAG_ENCODING/OUR_DATASET/top_5_testing_dataset.csv"

INSTRUCTION = (
    "Given a competitive programming problem, "
    "return the relevant algorithmic tags as a comma-separated list. "
    "Do not explain."
)

def clean_tags(tag_list) -> str:
    """Recebe uma lista de tags e retorna string limpa, separada por ', '."""
    if not isinstance(tag_list, list):
        return ""
    parts = []
    for t in tag_list:
        t = str(t).strip()
        if not t or t.startswith("*"):
            continue
        parts.append(t.lower())
    return ", ".join(parts)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\$\$\$.*?\$\$\$', ' ', text)
    text = re.sub(r'\$\$.*?\$\$', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\\[a-zA-Z]+\{[^}]*\}', ' ', text)
    text = re.sub(r'\\[a-zA-Z]+', ' ', text)
    text = re.sub(r'[{}]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def load_and_clean(path):
    df = pd.read_csv(path)
    # Se problem_tags estiver como string representando uma lista, converta
    if isinstance(df["problem_tags"].iloc[0], str) and df["problem_tags"].iloc[0].startswith("["):
        df["problem_tags"] = df["problem_tags"].apply(ast.literal_eval)
    df = df.dropna(subset=["problem_tags", "problem_statement"]).reset_index(drop=True)
    records = []
    for _, row in df.iterrows():
        text = clean_text(row["problem_statement"])
        tags = clean_tags(row["problem_tags"])
        if not text or not tags:
            continue
        records.append({
            "conversations": [
                {"role": "user", "content": f"{INSTRUCTION}\n\n{text}"},
                {"role": "assistant", "content": tags},
            ]
        })
    return records

# Carregar os três datasets
train_records = load_and_clean(TRAIN_PATH)
val_records   = load_and_clean(VAL_PATH)
test_records  = load_and_clean(TEST_PATH)

print(f"Treino: {len(train_records)} | Validação: {len(val_records)} | Teste: {len(test_records)}")

# Opcional: ver distribuição de tags no treino
all_tags = []
for r in train_records:
    all_tags.extend(r["conversations"][1]["content"].split(", "))
print(f"Top 20 tags (treino): {Counter(all_tags).most_common(20)}")

## Celula 5 -- Split treino/teste e apply_chat_template

Separamos 10% para avaliacao **antes** de aplicar o template,
mantendo `test_raw` com os campos originais para inferencia qualitativa.

`apply_chat_template` converte as conversations para o formato de texto
que o Qwen3 espera, adicionando os tokens `<|im_start|>` / `<|im_end|>`.

In [ ]:
# Função para aplicar o chat template (igual à original)
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
             for convo in convos]
    return {"text": texts}

# Dataset de treino (com template)
train_dataset = Dataset.from_list(train_records)
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

# Dataset de validação (com template)
val_dataset = Dataset.from_list(val_records)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Dataset de teste (mantido sem template, apenas como raw)
test_raw = Dataset.from_list(test_records)

print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_raw)}")

# Verificação rápida
print("\n--- Exemplo de prompt formatado (treino) ---")
print(train_dataset[0]["text"])

## Celula 6 -- Treino

### Por que `train_on_responses_only` resolve o bug do SFTTrainer
Nos notebooks anteriores usavamos `Trainer` base com label masking manual
porque o `SFTTrainer` sobrescrevia os labels calculados manualmente.
Aqui, o Unsloth aplica o masking **depois** que o SFTTrainer monta o dataset,
identificando os tokens de resposta pelo delimitador `<|im_start|>assistant`
e mascarando tudo antes dele com `-100`. O resultado e identico ao masking
manual, mas feito de forma segura e integrada.

| Parametro de treino | Valor | Obs |
|---|---|---|
| `per_device_train_batch_size` | 2 | seguro para 4B na T4/P100 |
| `gradient_accumulation_steps` | 8 | batch efetivo = 16 |
| `num_train_epochs` | 3 | -- |
| `learning_rate` | 2e-4 | padrao Unsloth |
| `optim` | `adamw_8bit` | menos VRAM |
| `fp16` | `True` | Kaggle T4/P100 nao suportam bf16 |
| `lr_scheduler_type` | `cosine` | decai suavemente ao longo das epocas |
| `warmup_ratio` | 0.05 | 5% dos steps de aquecimento |
| `weight_decay` | 0.01 | regularizacao leve |

> Estimativa: ~2-4 h no Kaggle com T4 x2 para 5 000 exemplos x 3 epocas.

In [ ]:
import os, torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Relatorio de VRAM antes de iniciar
gpu = torch.cuda.get_device_properties(0)
vram_total = gpu.total_memory / 1024**3
vram_used  = torch.cuda.memory_allocated(0) / 1024**3
print(f"GPU           : {gpu.name}")
print(f"VRAM total    : {vram_total:.1f} GB")
print(f"VRAM usada    : {vram_used:.1f} GB")
print(f"VRAM livre    : {vram_total - vram_used:.1f} GB")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,          # <--- adicionado
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs = 3,            # pode subir porque agora tem validação
        max_seq_length = MAX_SEQ_LENGTH,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        weight_decay = 0.01,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        seed = SEED,
        output_dir = "/kaggle/working/checkpoints",
        # --- OPÇÕES DE VALIDAÇÃO ---
        eval_strategy = "steps",   # avalia a cada eval_steps
        eval_steps = 50,                 # ajuste conforme o tamanho do dataset
        save_strategy = "steps",
        save_steps = 50,                 # alinhado com eval_steps
        save_total_limit = 3,
        load_best_model_at_end = True,   # carrega o melhor checkpoint automaticamente
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        # --------------------------
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Verificacao de masking: apenas a resposta deve aparecer, prompt vira espacos
sample_labels = trainer.train_dataset[0]["labels"]
sample_decoded = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample_labels]
).replace(tokenizer.pad_token, " ")
print("\n--- Labels mascarados (apenas a resposta deve aparecer) ---")
print(sample_decoded[:300])

## Celula 7 -- Iniciar treino

Para retomar apos interrupcao:
```python
trainer.train(resume_from_checkpoint=True)
```

In [ ]:
trainer_stats = trainer.train()

# Resumo de memoria e tempo
used_memory      = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
used_memory_lora = round(used_memory - vram_used, 2)
print(f"VRAM pico durante treino : {used_memory} GB")
print(f"VRAM consumida pelo LoRA : {used_memory_lora} GB")
print(f"Tempo total de treino    : {round(trainer_stats.metrics['train_runtime'] / 60, 1)} min")

## Celula 8 -- Inferencia

Ativa o kernel de inferencia otimizado do Unsloth (2x mais rapido).
Geracao **greedy** com parada em `<|im_end|>`.

> Aplique `clean_text()` no input antes de `predict_tags()` --
> o modelo foi treinado sem LaTeX e espera o mesmo pre-processamento.

In [ ]:
FastLanguageModel.for_inference(model)

IM_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
EOS_TOKEN_ID    = tokenizer.eos_token_id

def predict_tags(input_text: str) -> str:
    """input_text deve ter passado por clean_text() antes."""
    messages = [
        {"role": "user", "content": f"{INSTRUCTION}\n\n{input_text}"}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # adiciona <|im_start|>assistant
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[IM_END_TOKEN_ID, EOS_TOKEN_ID],
        )

    input_len  = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Teste rapido com o primeiro exemplo de teste
sample = test_raw[0]
true_tags = sample["conversations"][1]["content"]
pred_tags = predict_tags(clean_text(sample["conversations"][0]["content"].replace(INSTRUCTION + "\n\n", "")))
print("TRUE:", true_tags)
print("PRED:", pred_tags)

In [ ]:
import pandas as pd
import ast
from datasets import Dataset

# Caminho do seu dataset de avaliação externo
EXTERNAL_PATH = "/kaggle/input/datasets/dinuiongeorge/codeforces-competitive-programming-dataset/01_TASK_DATASETS/03_Task_Datasets/02_DATASETS_WO_TAG_ENCODING/OUR_DATASET/top_5_validation_dataset.csv"

df_ext = pd.read_csv(EXTERNAL_PATH)

# Se problem_tags estiver como string-a-ser-avaliada em lista
if df_ext["problem_tags"].iloc[0].startswith("["):
    df_ext["problem_tags"] = df_ext["problem_tags"].apply(ast.literal_eval)

df_ext = df_ext.dropna(subset=["problem_tags", "problem_statement"]).reset_index(drop=True)

# ----- As funções de limpeza DEVEM ser as mesmas do treino -----
def clean_tags_ext(tag_list) -> str:
    if not isinstance(tag_list, list):
        return ""
    parts = []
    for t in tag_list:
        t = str(t).strip()
        if not t or t.startswith("*"):
            continue
        parts.append(t.lower())
    return ", ".join(parts)

def clean_text_ext(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\$\$\$.*?\$\$\$', ' ', text)
    text = re.sub(r'\$\$.*?\$\$', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\\[a-zA-Z]+\{[^}]*\}', ' ', text)
    text = re.sub(r'\\[a-zA-Z]+', ' ', text)
    text = re.sub(r'[{}]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
# ----------------------------------------------------------------

records_ext = []
for _, row in df_ext.iterrows():
    text = clean_text_ext(row["problem_statement"])
    tags = clean_tags_ext(row["problem_tags"])
    if not text or not tags:
        continue
    records_ext.append({
        "conversations": [
            {"role": "user",      "content": f"{INSTRUCTION}\n\n{text}"},
            {"role": "assistant", "content": tags},
        ]
    })

# Converte para Dataset (formato usado na avaliação original)
eval_dataset = Dataset.from_list(records_ext)
print(f"Exemplos no dataset externo: {len(eval_dataset)}")

## Celula 9 -- Avaliacao completa

Metricas multilabel sobre ate 200 exemplos do conjunto de teste:

| Metrica | Descricao |
|---|---|
| F1 Micro | F1 ponderado por frequencia de tag |
| F1 Macro | F1 medio por tag (ignora frequencia) |
| Precision / Recall | Micro-averaged |
| Exact Match | Fracao com conjunto de tags identico |
| Coverage | Fracao das tags verdadeiras previstas |
| Overprediction | Fracao onde o modelo previu tags a mais |

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

eval_dataset = test_raw               # <--- mudança aqui
EVAL_N = len(eval_dataset)            # avalia todos os exemplos

def parse_tags(tag_string: str) -> set:
    if not isinstance(tag_string, str):
        return set()
    return {t.strip().lower() for t in tag_string.split(",") if t.strip()}

y_true, y_pred = [], []
precisions, recalls, exact_matches = [], [], []
num_true_tags, num_pred_tags = [], []

print(f"Avaliando {EVAL_N} exemplos...")

for i, example in enumerate(eval_dataset.select(range(EVAL_N))):
    raw_input = example["conversations"][0]["content"].replace(INSTRUCTION + "\n\n", "")
    true_tags = parse_tags(example["conversations"][1]["content"])
    pred_text = predict_tags(clean_text(raw_input))
    pred_tags = parse_tags(pred_text)

    y_true.append(true_tags)
    y_pred.append(pred_tags)

    prec = len(true_tags & pred_tags) / len(pred_tags) if pred_tags else 0.0
    rec  = len(true_tags & pred_tags) / len(true_tags) if true_tags else 1.0
    precisions.append(prec)
    recalls.append(rec)
    exact_matches.append(int(true_tags == pred_tags))
    num_true_tags.append(len(true_tags))
    num_pred_tags.append(len(pred_tags))

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{EVAL_N}")

all_tags_eval = sorted(set().union(*y_true, *y_pred))

def to_binary(tag_sets, tags):
    return [[1 if t in ts else 0 for t in tags] for ts in tag_sets]

y_true_bin = to_binary(y_true, all_tags_eval)
y_pred_bin = to_binary(y_pred, all_tags_eval)

f1_micro   = f1_score(y_true_bin, y_pred_bin, average="micro",  zero_division=0)
f1_macro   = f1_score(y_true_bin, y_pred_bin, average="macro",  zero_division=0)
prec_micro = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
rec_micro  = recall_score(y_true_bin,  y_pred_bin, average="micro", zero_division=0)
exact_match = float(np.mean(exact_matches))
overpred    = float(np.mean([p > t for p, t in zip(num_pred_tags, num_true_tags)]))
coverage    = float(np.mean([
    len(tr & pr) / len(tr) if tr else 1.0
    for tr, pr in zip(y_true, y_pred)
]))

missed = Counter()
extra  = Counter()
for true, pred in zip(y_true, y_pred):
    for t in true - pred: missed[t] += 1
    for t in pred - true: extra[t]  += 1

never_predicted = set(all_tags_eval) - {t for tags in y_pred for t in tags}

print("\n" + "="*50)
print(f"F1 Micro      : {f1_micro:.4f}")
print(f"F1 Macro      : {f1_macro:.4f}")
print(f"Precision     : {prec_micro:.4f}")
print(f"Recall        : {rec_micro:.4f}")
print(f"Exact Match   : {exact_match:.4f}")
print(f"Coverage      : {coverage:.4f}")
print(f"Overprediction: {overpred:.4f}")
print(f"Avg true tags : {np.mean(num_true_tags):.2f}")
print(f"Avg pred tags : {np.mean(num_pred_tags):.2f}")
print("\nTags mais perdidas  :", missed.most_common(10))
print("Tags mais inventadas:", extra.most_common(10))

print("\n=== EXEMPLOS ===")
for i in range(min(10, EVAL_N)):
    print(f"\n--- {i+1} ---")
    print(f"TRUE : {sorted(y_true[i])}")
    print(f"PRED : {sorted(y_pred[i])}")

## Celula 10 -- Salvar modelo e resultados

No Kaggle, o output persistente fica em `/kaggle/working/`.
O adaptador LoRA (~80 MB para 4B) e salvo la e fica disponivel
para download ou para ser carregado em outros notebooks.

In [ ]:
import os, json
from datetime import datetime
from huggingface_hub import login, HfApi, create_repo

# ---- Configuração ----
EXPERIMENT_NAME = "qwen3_4b_instruct_2507_qlora_v2"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE = "/kaggle/working"
HF_REPO_ID = "Milangs/qwen3-4b-tag-predictor"   # <-- COLOQUE SEU NOME DE USUÁRIO CORRETO

# ---- 1. Login no Hugging Face ----
from kaggle_secrets import UserSecretsClient
secret_label = "hftoken"
secret_value = UserSecretsClient().get_secret(secret_label)
if not secret_value:
    raise ValueError("Token HF não encontrado! Adicione o secret HF_TOKEN nas configurações do Kaggle.")
login(token=secret_value)

# ---- 2. Salvar modelo e tokenizador localmente ----
model_dir = f"{OUTPUT_BASE}/models/{EXPERIMENT_NAME}_{TIMESTAMP}/lora_adapter"
os.makedirs(model_dir, exist_ok=True)
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"Modelo salvo localmente em: {model_dir}")

# ---- 3. Criar repositório no Hub (se não existir) ----
try:
    create_repo(repo_id=HF_REPO_ID, private=False, exist_ok=True)
    print(f"Repositório {HF_REPO_ID} pronto.")
except Exception as e:
    print(f"Aviso ao criar repo: {e}")

# ---- 4. Enviar modelo e tokenizador para o Hub ----
model.push_to_hub(
    repo_id=HF_REPO_ID,
    token=secret_value,
    commit_message=f"Upload LoRA adapter – {TIMESTAMP}"
)
tokenizer.push_to_hub(
    repo_id=HF_REPO_ID,
    token=secret_value,
    commit_message=f"Upload tokenizer – {TIMESTAMP}"
)
print(f"✅ Modelo enviado com sucesso para: https://huggingface.co/{HF_REPO_ID}")

# ---- 5. Construir e salvar métricas ----
# Garante que as variáveis da avaliação (célula 9) estão disponíveis
if all(var in globals() for var in ["EVAL_N", "f1_micro", "f1_macro", "prec_micro", "rec_micro", 
                                     "precisions", "recalls", "exact_match", "num_true_tags", 
                                     "num_pred_tags", "overpred", "coverage", "all_tags_eval", 
                                     "never_predicted", "missed", "extra"]):
    
    import numpy as np  # já deve estar importado, mas por segurança

    results = {
        "experiment":   EXPERIMENT_NAME,
        "timestamp":    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "eval_samples": EVAL_N,
        "model": {
            "base":           "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
            "lora_r":         32,
            "lora_alpha":     32,
            "lora_dropout":   0,
            "max_seq_length": MAX_SEQ_LENGTH,
            "train_samples":  len(train_records),
            "epochs":         3,
            "learning_rate":  2e-4,
        },
        "metrics": {
            "f1_micro":        round(f1_micro,   4),
            "f1_macro":        round(f1_macro,   4),
            "precision_micro": round(prec_micro, 4),
            "recall_micro":    round(rec_micro,  4),
            "precision_mean":  round(float(np.mean(precisions)), 4),
            "recall_mean":     round(float(np.mean(recalls)),    4),
            "exact_match":     round(exact_match, 4),
        },
        "stats": {
            "avg_true_tags":         round(float(np.mean(num_true_tags)), 2),
            "avg_pred_tags":         round(float(np.mean(num_pred_tags)), 2),
            "overprediction":        round(overpred,  4),
            "coverage":              round(coverage,  4),
            "unique_tags_in_eval":   len(all_tags_eval),
            "never_predicted_count": len(never_predicted),
            "never_predicted_tags":  sorted(list(never_predicted))[:30],
        },
        "errors": {
            "most_missed": missed.most_common(10),
            "most_extra":  extra.most_common(10),
        },
    }

    # Salvar localmente
    results_dir = f"{OUTPUT_BASE}/results"
    os.makedirs(results_dir, exist_ok=True)
    results_path = f"{results_dir}/{EXPERIMENT_NAME}.json"
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Resultados salvos em: {results_path}")

    # Fazer upload do JSON de métricas para o Hub
    api = HfApi()
    api.upload_file(
        path_or_fileobj=results_path,
        path_in_repo="metrics.json",
        repo_id=HF_REPO_ID,
        token=hf_token,
    )
    print("Métricas enviadas como metrics.json")
else:
    print("⚠️ Variáveis de avaliação não encontradas. Execute a Célula 9 antes desta.")

In [ ]:
import os
import subprocess
from IPython.display import FileLink, display

# Garantir que as variáveis estão definidas (caso a célula rode sozinha)
try:
    EXPERIMENT_NAME
    TIMESTAMP
except NameError:
    # Se não definidas, defina aqui os mesmos valores da Célula 10
    from datetime import datetime
    EXPERIMENT_NAME = "qwen3_4b_instruct_2507_qlora_v2"
    TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))

# Baixar o arquivo JSON de resultados
download_file(f"results/{EXPERIMENT_NAME}.json", "resultados_avaliacao")

# Baixar o adaptador LoRA (pasta completa)
#download_file(f"models/{EXPERIMENT_NAME}_{TIMESTAMP}/lora_adapter", "modelo_lora_adapter")